### Cell 1：导入模块和控件

In [ ]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import sympy as sp


# 从模块导入
from linear_solvers import lu_solve, jacobi_iteration, gauss_seidel_iteration, conjugate_gradient
from equations import f, df, phi, generate_matrix
from nonlinear_solvers import (
    simple_iteration,
    steffensen,
    newton,
    newton_downhill
)
from eigen_solvers import power_method, inverse_power_method

### Cell 2：非线性方程求根控件

In [ ]:
# 公共参数控件
x0_slider = widgets.FloatText(value=1.0, min=0.0, max=2.0, step=0.01, description='初值 x0:')
eps_text = widgets.BoundedFloatText(value=1e-6, min=1e-15, max=1e-1, step=1e-7, description='误差限:', format='.1e')
max_iter_slider = widgets.IntText(value=100, min=10, max=500, description='最大迭代次数:')

# 牛顿下山法专用控件
lambda_eps = widgets.BoundedFloatText(value=1e-8, min=1e-15, max=1e-1, description='下山下限:', format='.1e')
max_downhill_slider = widgets.IntText(value=20, min=5, max=50, description='最大下山次数:')

# 方法选择
method_selector = widgets.Dropdown(
    options=['简单迭代法', 'Steffensen法', '牛顿迭代法', '牛顿下山法'],
    value='简单迭代法',
    description='方法:'
)

run_button = widgets.Button(description='开始求根', button_style='success')
output_area = widgets.Output()

# ==================== 方程自定义控件 ====================
equation_mode = widgets.RadioButtons(
    options=['默认方程', '自定义方程'],
    value='默认方程',
    description='方程来源:'
)

# 自定义方程输入框
f_expr_text = widgets.Text(
    value='x**3 + 2*x**2 + 10*x - 20',
    description='f(x)=',
    layout=widgets.Layout(width='400px')
)
phi_expr_text = widgets.Text(
    value='20 / (x**2 + 2*x + 10)',
    description='φ(x)=',
    layout=widgets.Layout(width='400px')
)
df_expr_text = widgets.Text(
    value='',  # 留空则自动求导
    placeholder='留空则自动求导',
    description='f\'(x)=',
    layout=widgets.Layout(width='400px')
)

parse_eq_button = widgets.Button(description='解析并更新方程', button_style='warning')
eq_status = widgets.Output()

# 存储当前使用的函数（默认指向 equations.py 中的函数）
current_f = f
current_df = df
current_phi = phi

def update_eq_status():
    with eq_status:
        clear_output()
        print("当前使用方程：")
        print(f"  f(x) = {f_expr_text.value}")
        print(f"  φ(x) = {phi_expr_text.value}")
        if df_expr_text.value.strip():
            print(f"  f'(x) = {df_expr_text.value.strip()}")
        else:
            print("  f'(x) = 自动求导")

def on_parse_eq(b):
    global current_f, current_df, current_phi
    x = sp.Symbol('x')
    try:
        # 解析 f(x)
        f_expr = sp.sympify(f_expr_text.value)
        f_func = sp.lambdify(x, f_expr, 'numpy')

        # 解析或自动求导 f'(x)
        if df_expr_text.value.strip():
            df_expr = sp.sympify(df_expr_text.value)
            df_func = sp.lambdify(x, df_expr, 'numpy')
        else:
            df_expr = sp.diff(f_expr, x)
            df_func = sp.lambdify(x, df_expr, 'numpy')
            # 更新文本框显示自动求导结果（可选）
            df_expr_text.value = str(df_expr)

        # 解析 φ(x)
        phi_expr = sp.sympify(phi_expr_text.value)
        phi_func = sp.lambdify(x, phi_expr, 'numpy')

        # 更新全局函数
        current_f = f_func
        current_df = df_func
        current_phi = phi_func

        with eq_status:
            clear_output()
            print("✅ 方程解析成功！")
            print(f"  f(x) = {f_expr}")
            print(f"  φ(x) = {phi_expr}")
            print(f"  f'(x) = {df_expr}")
    except Exception as e:
        with eq_status:
            clear_output()
            print(f"❌ 解析失败: {e}")

def on_eq_mode_change(change):
    if change['new'] == '默认方程':
        # 切回默认函数
        global current_f, current_df, current_phi
        from equations import f as default_f, df as default_df, phi as default_phi
        current_f = default_f
        current_df = default_df
        current_phi = default_phi
        # 隐藏自定义输入框
        f_expr_text.layout.visibility = 'hidden'
        phi_expr_text.layout.visibility = 'hidden'
        df_expr_text.layout.visibility = 'hidden'
        parse_eq_button.layout.visibility = 'hidden'
        with eq_status:
            clear_output()
            print("当前使用默认方程：f(x) = x^3 + 2x^2 + 10x - 20")
    else:
        f_expr_text.layout.visibility = 'visible'
        phi_expr_text.layout.visibility = 'visible'
        df_expr_text.layout.visibility = 'visible'
        parse_eq_button.layout.visibility = 'visible'
        update_eq_status()

equation_mode.observe(on_eq_mode_change, names='value')
parse_eq_button.on_click(on_parse_eq)

# 界面组合
equation_box = widgets.VBox([
    equation_mode,
    widgets.HBox([f_expr_text, phi_expr_text]),
    df_expr_text,
    parse_eq_button,
    eq_status
])

# 初始状态（默认方程模式）
f_expr_text.layout.visibility = 'hidden'
phi_expr_text.layout.visibility = 'hidden'
df_expr_text.layout.visibility = 'hidden'
parse_eq_button.layout.visibility = 'hidden'

### Cell 3：求根事件响应函数

In [ ]:
def on_run_clicked(b):
    with output_area:
        clear_output()
        x0 = x0_slider.value
        eps = eps_text.value
        max_iter = max_iter_slider.value
        method = method_selector.value

        # 使用当前激活的方程函数
        f_now = current_f
        df_now = current_df
        phi_now = current_phi

        try:
            if method == '简单迭代法':
                root, iters, conv = simple_iteration(phi_now, x0, eps, max_iter)
                print(f"【{method}】")
                print(f"近似根: {root:.10f}")
                print(f"迭代次数: {iters}, 收敛情况: {'收敛' if conv else '未收敛'}")
                print(f"f(root) = {f_now(root):.2e}")
            elif method == 'Steffensen法':
                root, iters, conv = steffensen(phi_now, x0, eps, max_iter)
                print(f"【{method}】")
                print(f"近似根: {root:.10f}")
                print(f"迭代次数: {iters}, 收敛情况: {'收敛' if conv else '未收敛'}")
                print(f"f(root) = {f_now(root):.2e}")
            elif method == '牛顿迭代法':
                root, iters, conv = newton(f_now, df_now, x0, eps, max_iter)
                print(f"【{method}】")
                print(f"近似根: {root:.10f}")
                print(f"迭代次数: {iters}, 收敛情况: {'收敛' if conv else '未收敛'}")
                print(f"f(root) = {f_now(root):.2e}")
            elif method == '牛顿下山法':
                root, iters, lambdas, conv = newton_downhill(
                    f_now, df_now, x0, eps, max_iter,
                    epsilon_lambda=lambda_eps.value,
                    max_downhill=max_downhill_slider.value
                )
                print(f"【{method}】")
                print(f"近似根: {root:.10f}")
                print(f"迭代次数: {iters}, 收敛情况: {'收敛' if conv else '未收敛'}")
                print(f"f(root) = {f_now(root):.2e}")
                print(f"各步下山因子: {np.round(lambdas, 6).tolist()}")
        except Exception as e:
            print(f"计算出错：{e}")

run_button.on_click(on_run_clicked)

### Cell 4：非线性方程求根界面布局

In [ ]:
nonlinear_box = widgets.VBox([
    equation_box,
    widgets.HBox([x0_slider, eps_text]),
    widgets.HBox([max_iter_slider, method_selector]),
    widgets.HBox([lambda_eps, max_downhill_slider]),
    run_button,
    output_area
])

### Cell 5：矩阵特征值控件

In [ ]:
# 矩阵输入方式选择
input_mode = widgets.RadioButtons(
    options=['随机生成', '手动输入'],
    value='随机生成',
    description='矩阵来源:'
)

# 随机生成控件
size_input = widgets.IntText(value=4, description='矩阵大小:')
seed_text = widgets.IntText(value=42, description='随机种子:')
regenerate_button = widgets.Button(description='生成随机矩阵')

# 手动输入控件
matrix_textarea = widgets.Textarea(
    value='1 2 3 4;\n5 6 7 8;\n9 10 11 12;\n13 14 15 16',
    placeholder='输入矩阵，例如：1 2;3 4 或 1,2,3;4,5,6',
    description='矩阵:',
    layout=widgets.Layout(width='400px', height='100px')
)
parse_button = widgets.Button(description='解析矩阵', button_style='info')

# 特征值方法选择
eigen_method_selector = widgets.Dropdown(
    options=['幂法', '逆幂法'],
    value='幂法',
    description='方法:'
)
run_eigen_button = widgets.Button(description='开始计算特征值', button_style='success')
eigen_output = widgets.Output()

# 当前矩阵变量
current_A = generate_matrix(4, 42)
matrix_display = widgets.Output()

### Cell 6：矩阵生成与事件函数

In [ ]:
def update_matrix_display():
    with matrix_display:
        clear_output()
        print("当前矩阵 A =")
        print(np.round(current_A, 4))

def on_regenerate(b):
    global current_A
    if seed_text.value > 0:
        current_A = generate_matrix(size_input.value, seed_text.value)
    else:
        current_A = generate_matrix(size_input.value)
    update_matrix_display()

def parse_matrix_from_text(text):
    """从字符串解析矩阵，支持空格、逗号、分号、换行分隔"""
    # 按分号或换行切分行
    rows = text.replace(';', '\n').strip().split('\n')
    data = []
    for row in rows:
        row = row.strip()
        if not row:
            continue
        # 用逗号或空格分割
        parts = row.replace(',', ' ').split()
        if not parts:
            continue
        # 转换为浮点数
        numbers = [float(x) for x in parts]
        data.append(numbers)
    if not data:
        raise ValueError("矩阵为空，请检查输入格式")
    # 检查每行列数一致
    ncol = len(data[0])
    for i, row in enumerate(data):
        if len(row) != ncol:
            raise ValueError(f"第 {i+1} 行元素个数不匹配（应为 {ncol}，实际 {len(row)}）")
    return np.array(data)

def on_parse(b):
    global current_A
    with eigen_output:
        clear_output()
        try:
            current_A = parse_matrix_from_text(matrix_textarea.value)
            # 方阵检查
            if current_A.shape[0] != current_A.shape[1]:
                print("矩阵不是方阵（行数 ≠ 列数），特征值方法需要方阵。")
                return
            update_matrix_display()
            print("矩阵解析成功，已更新。")
        except Exception as e:
            print(f"解析失败：{e}")

def on_mode_change(change):
    # 根据选择显示/隐藏相关控件
    if change['new'] == '随机生成':
        size_input.layout.visibility = 'visible'
        seed_text.layout.visibility = 'visible'
        regenerate_button.layout.visibility = 'visible'
        matrix_textarea.layout.visibility = 'hidden'
        parse_button.layout.visibility = 'hidden'
    else:
        size_input.layout.visibility = 'hidden'
        seed_text.layout.visibility = 'hidden'
        regenerate_button.layout.visibility = 'hidden'
        matrix_textarea.layout.visibility = 'visible'
        parse_button.layout.visibility = 'visible'

input_mode.observe(on_mode_change, names='value')

def on_run_eigen(b):
    with eigen_output:
        clear_output()
        A = current_A
        eps = eps_text.value
        max_iter = max_iter_slider.value
        method = eigen_method_selector.value

        try:
            if method == '幂法':
                val, vec, iters, conv = power_method(A, epsilon=eps, max_iter=max_iter)
                print(f"【幂法：主特征值】")
            else:
                # 尝试求逆，若矩阵奇异则给出提示
                if np.linalg.matrix_rank(A) < A.shape[0]:
                    print("矩阵奇异，无法使用逆幂法（需要可逆矩阵）")
                    return
                val, vec, iters, conv = inverse_power_method(A, epsilon=eps, max_iter=max_iter)
                print(f"【逆幂法：最小特征值】")
            print(f"特征值: {val:.10f}")
            print(f"迭代次数: {iters}, 收敛: {'收敛' if conv else '未收敛'}")
            print(f"特征向量: {np.round(vec, 6)}")

            # 矩阵条件数与残差检查（检查矩阵是否非奇异）
            cond = np.linalg.cond(A)
            if cond > 1e10:
                print(f"⚠️ 矩阵条件数 = {cond:.2e}（很大），结果可能不可靠")

            residual = np.linalg.norm(A @ vec - val * vec)
            print(f"残差 ||A·vec - λ·vec|| = {residual:.2e}")
            if residual > 1e-4:
                print("⚠️ 残差较大，检查特征值/向量是否正确")

            # 对比 numpy 真实特征值
            eigvals, _ = np.linalg.eig(A)
            print(f"\nnumpy 全部特征值: {np.round(eigvals, 6)}")
            if method == '幂法':
                idx = np.argmax(np.abs(eigvals))
            else:
                idx = np.argmin(np.abs(eigvals))
            print(f"参考值: {eigvals[idx]:.6f}")
        except np.linalg.LinAlgError:
            print("矩阵奇异，无法求逆（逆幂法需要可逆矩阵）")
        except Exception as e:
            print(f"计算错误：{e}")

# 绑定按钮事件
regenerate_button.on_click(on_regenerate)
parse_button.on_click(on_parse)
run_eigen_button.on_click(on_run_eigen)

# 初始显示
update_matrix_display()
# 默认随机生成模式，手动输入控件隐藏
matrix_textarea.layout.visibility = 'hidden'
parse_button.layout.visibility = 'hidden'

### Cell 7：特征值界面布局

In [ ]:
eigen_box = widgets.VBox([
    input_mode,
    widgets.HBox([size_input, seed_text, regenerate_button]),
    widgets.HBox([matrix_textarea, parse_button]),
    matrix_display,
    widgets.HBox([eigen_method_selector, run_eigen_button]),
    eigen_output
])

### Cell 8：线性求解单元

In [ ]:
# ==================== 线性方程组专属 UI 控件 ====================
linear_matrix_textarea = widgets.Textarea(
    value='5 -1 -1 0;\n-1 5 0 -1;\n-1 0 5 -1;\n0 -1 -1 5',
    placeholder='输入方阵A，如: 2 1; 1 3',
    description='矩阵 A:',
    layout=widgets.Layout(width='400px', height='100px')
)
linear_b_textarea = widgets.Text(
    value='3 3 3 3',
    placeholder='输入向量b，用空格或逗号分隔',
    description='向量 b:',
    layout=widgets.Layout(width='400px')
)
linear_method_selector = widgets.Dropdown(
    options=['LU分解法', 'Jacobi迭代法', 'Gauss-Seidel迭代法', '共轭梯度法(CG)'],
    value='LU分解法',
    description='求解方法:'
)
run_linear_button = widgets.Button(description='开始求解方程组', button_style='primary')
linear_output = widgets.Output()

def parse_vector_from_text(text):
    parts = text.replace(',', ' ').split()
    return np.array([float(x) for x in parts], dtype=np.float64)

def on_run_linear_clicked(b_trigger):
    with linear_output:
        clear_output()
        try:
            A = parse_matrix_from_text(linear_matrix_textarea.value)
            b = parse_vector_from_text(linear_b_textarea.value)
            eps = eps_text.value
            max_iter = max_iter_slider.value
            method = linear_method_selector.value

            if A.shape[0] != A.shape[1]:
                print("错误：矩阵 A 不是方阵！")
                return
            if A.shape[0] != len(b):
                print(f"错误：矩阵行数与向量b维度不匹配！")
                return

            print(f"【{method} 求解中...】")

            if method == 'LU分解法':
                x = lu_solve(A, b)
                iters, conv = "N/A", True
            elif method == 'Jacobi迭代法':
                x, iters, conv = jacobi_iteration(A, b, tol=eps, max_iter=max_iter)
            elif method == 'Gauss-Seidel迭代法':
                x, iters, conv = gauss_seidel_iteration(A, b, tol=eps, max_iter=max_iter)
            elif method == '共轭梯度法(CG)':
                x, iters, conv = conjugate_gradient(A, b, tol=eps, max_iter=max_iter)

            print(f"计算解 x = {np.round(x, 6)}")
            print(f"迭代次数: {iters}, 收敛状态: {conv}")

            residual = np.linalg.norm(A @ x - b)
            print(f"后验残差 ||Ax - b|| = {residual:.2e}")

        except Exception as e:
            print(f"计算过程中发生错误: {e}")

run_linear_button.on_click(on_run_linear_clicked)

linear_box = widgets.VBox([
    widgets.HBox([linear_matrix_textarea, linear_b_textarea]),
    widgets.HBox([linear_method_selector, run_linear_button]),
    linear_output
])

### Cell 9:插值法与曲线拟合

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from equations import runge_function, generate_chebyshev_nodes
from interpolation_solvers import lagrange_interpolation, newton_interpolation, best_square_approximation_cubic


# 1. 自定义函数与区间控件
func_input = widgets.Text(
    value='1/(1+9*x**2)',
    description='函数 f(x):',
    placeholder='例如: sin(x) 或 1/(1+9*x**2)',
    layout=widgets.Layout(width='400px')
)

interval_a = widgets.FloatText(
    value=-1.0,
    description='区间下限 a:',
    layout=widgets.Layout(width='198px')
)

interval_b = widgets.FloatText(
    value=1.0,
    description='区间上限 b:',
    layout=widgets.Layout(width='198px')
)
interval_box = widgets.HBox([interval_a, interval_b])

# 2. 节点与选项控件
interp_nodes_slider = widgets.IntSlider(
    value=10, min=4, max=25, step=1,
    description='节点数 n:', layout=widgets.Layout(width='400px')
)

interp_type_selector = widgets.Dropdown(
    options=['等分节点', 'Chebyshev零点'],
    value='等分节点',
    description='节点分布:',
    layout=widgets.Layout(width='400px')
)

run_interp_button = widgets.Button(
    description='生成高级对比曲线图',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='400px', margin='10px 0px 0px 0px')
)

interp_output = widgets.Output()

# 3. 核心中控计算与绘图事件
def on_run_interp_clicked(b_trigger):
    with interp_output:
        clear_output(wait=True)
        
        # 强行注入局部抗乱码字体
        plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
        plt.rcParams['axes.unicode_minus'] = False
        
        # 获取前端输入的动态参数
        func_str = func_input.value.strip()
        a = interval_a.value
        b = interval_b.value
        n = interp_nodes_slider.value
        node_type = interp_type_selector.value
        
        # 异常防御
        if a >= b:
            print("【错误】区间下限 a 必须小于上限 b!")
            return
            
        x = sp.Symbol('x')
        try:
            f_expr = sp.sympify(func_str) 
            f_numpy = sp.lambdify(x, f_expr, 'numpy')
            f_numpy(np.array([a, (a+b)/2, b])) # 探测定义域
        except Exception as e:
            print(f"【解析失败】无法识别表达式，或函数在区间内无定义！")
            print(f"请检查语法：例如乘号不可省略，必须写成 9*x**2 而不是 9x^2")
            return

        # 生成插值节点
        if node_type == '等分节点':
            x_nodes = np.linspace(a, b, n)
        else:
            # 切比雪夫任意区间仿射变换换元
            i = np.arange(1, n + 1)
            t_nodes = np.cos((2 * i - 1) / (2 * n) * np.pi)
            x_nodes = 0.5 * (b - a) * t_nodes + 0.5 * (a + b)
            x_nodes = np.sort(x_nodes)
            
        y_nodes = f_numpy(x_nodes)
        
        # 绘图评估点
        x_dense = np.linspace(a, b, 250)
        y_true = f_numpy(x_dense)
        
        # 底层计算内核调用
        try:
            y_lagrange = lagrange_interpolation(x_nodes, y_nodes, x_dense)
            y_newton = newton_interpolation(x_nodes, y_nodes, x_dense)
            
            # 通用三次最佳平方逼近符号推导
            phi = [1, x, x**2, x**3]
            H = sp.Matrix.zeros(4, 4)
            B = sp.Matrix.zeros(4, 1)
            for r in range(4):
                for c in range(4):
                    H[r, c] = sp.integrate(phi[r] * phi[c], (x, a, b))
                B[r, 0] = sp.integrate(f_expr * phi[r], (x, a, b))
            
            coef_c = H.LUsolve(B)
            poly_expr = sp.simplify(sum(coef_c[k] * phi[k] for k in range(4)))
            poly_func = sp.lambdify(x, poly_expr, 'numpy')
            y_cubic = poly_func(x_dense)
            
        except Exception as e:
            print(f"【计算崩溃】积分或求解失败（可能存在发散的瑕积分或条件数极高）: {e}")
            return
        
        # 渲染图像
        plt.figure(figsize=(10, 6))
        plt.plot(x_dense, y_true, 'k--', linewidth=2, label=f'原函数 $f(x)$')
        plt.plot(x_dense, y_lagrange, 'r-', label=f'Lagrange 插值 (n={n})')
        plt.plot(x_dense, y_newton, 'b-.', label=f'Newton 插值 (n={n})')
        plt.plot(x_dense, y_cubic, 'g-', linewidth=2, label='三次最佳平方逼近')
        plt.scatter(x_nodes, y_nodes, color='magenta', zorder=5, label='当前计算节点')
        
        plt.title(f'函数在 [{a}, {b}] 上的插值与逼近对比 ({node_type})', fontsize=12, fontweight='bold')
        plt.xlabel('x')
        plt.ylabel('y')
        plt.xlim(a - (b-a)*0.05, b + (b-a)*0.05)
        # 将图例移到图像右侧外部，对齐顶部
        plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
        # 自动调整画布边缘，防止外部图例被网页切掉
        plt.tight_layout()
        plt.grid(True, linestyle=':')
        plt.show()
        
        print(f"【推导结果】")
        print(f"该函数在区间 [{a}, {b}] 上的三次最佳平方逼近多项式解析式为：")
        display(poly_expr)

run_interp_button.on_click(on_run_interp_clicked)

# 将第四模块组装成一个面板 (Box)
interp_box = widgets.VBox([
    widgets.HTML("<h4>🎨 函数与参数设定</h4>"),
    func_input,
    interval_box,
    interp_nodes_slider,
    interp_type_selector,
    run_interp_button,
    widgets.HTML("<hr>"),
    interp_output
])





### Cell 10：全局布局

In [ ]:
config_box = widgets.HBox([
    widgets.HTML("<b>全局控制：</b>"),
    eps_text, 
    max_iter_slider
])

main_tab = widgets.Tab()
#前面三个实验生成的变量
main_tab.children = [nonlinear_box, eigen_box, linear_box, interp_box]

main_tab.set_title(0, '① 非线性方程求根')
main_tab.set_title(1, '② 矩阵特征值计算')
main_tab.set_title(2, '③ 线性方程组求解')
main_tab.set_title(3, '④ 通用插值与逼近平台') 

# 最终渲染仪表盘
dashboard = widgets.VBox([
    config_box,  # 最顶部的全局误差与迭代次数配置区
    widgets.HTML("<hr>"),
    main_tab
])

display(dashboard)